In [3]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("Lab2-Transactions") #konwencja kropka na poczatku
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version} — gotowy")

Spark 4.0.0-preview2 — gotowy


In [4]:
df = spark.read.json("transactions_10k.jsonl")

print(f"Liczba rekordów: {df.count()}")
df.printSchema()

Liczba rekordów: 10000
root
 |-- amount: double (nullable = true)
 |-- category: string (nullable = true)
 |-- store: string (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- tx_id: string (nullable = true)
 |-- user_id: string (nullable = true)



In [5]:
# ZADANIE DOMOWE 
# 1. Najniższa średnia kwota – sklep Gdańsk
from pyspark.sql.functions import window, col, avg

gdn_avg = (
    df
    .filter(col("store") == "Gdańsk")
    .groupBy(window("timestamp", "1 hour"))
    .agg(avg("amount").alias("avg_amount"))
)

lowest_hour = gdn_avg.orderBy("avg_amount").limit(1)

lowest_hour.show(truncate=False)

+------------------------------------------+-----------------+
|window                                    |avg_amount       |
+------------------------------------------+-----------------+
|{2026-04-12 08:00:00, 2026-04-12 09:00:00}|395.0118407310706|
+------------------------------------------+-----------------+



In [6]:
# 2. Liczba transakcji per kategoria (09:00–09:30)

from pyspark.sql.functions import count, hour, minute, col

transactions_0930 = (
    df
    .filter(
        (hour(col("timestamp")) == 9) &
        (minute(col("timestamp")) < 30)
    )
    .groupBy("category")
    .agg(count("tx_id").alias("tx_count"))
    .orderBy("category")
)

transactions_0930.show(truncate=False)

+-----------+--------+
|category   |tx_count|
+-----------+--------+
|elektronika|611     |
|książki    |622     |
|odzież     |605     |
|żywność    |567     |
+-----------+--------+



In [7]:
# 3. Okna 15-minutowe – szczyt transakcji
from pyspark.sql.functions import window, count, col

peak_15min = (
    df
    .groupBy(window("timestamp", "15 minutes"))
    .agg(count("tx_id").alias("tx_count"))
    .orderBy(col("tx_count").desc())
)

peak_15min.show(truncate=False)

+------------------------------------------+--------+
|window                                    |tx_count|
+------------------------------------------+--------+
|{2026-04-12 09:15:00, 2026-04-12 09:30:00}|1234    |
|{2026-04-12 09:00:00, 2026-04-12 09:15:00}|1171    |
|{2026-04-12 09:30:00, 2026-04-12 09:45:00}|1156    |
|{2026-04-12 08:45:00, 2026-04-12 09:00:00}|1139    |
|{2026-04-12 09:45:00, 2026-04-12 10:00:00}|1100    |
|{2026-04-12 08:30:00, 2026-04-12 08:45:00}|899     |
|{2026-04-12 10:00:00, 2026-04-12 10:15:00}|858     |
|{2026-04-12 08:15:00, 2026-04-12 08:30:00}|644     |
|{2026-04-12 10:15:00, 2026-04-12 10:30:00}|582     |
|{2026-04-12 08:00:00, 2026-04-12 08:15:00}|468     |
|{2026-04-12 10:30:00, 2026-04-12 10:45:00}|443     |
|{2026-04-12 10:45:00, 2026-04-12 11:00:00}|306     |
+------------------------------------------+--------+



In [8]:
peak_15min.limit(1).show(truncate=False)

+------------------------------------------+--------+
|window                                    |tx_count|
+------------------------------------------+--------+
|{2026-04-12 09:15:00, 2026-04-12 09:30:00}|1234    |
+------------------------------------------+--------+

